# Get the playlist downloaded

In [3]:
from pathlib import Path

home_music_dir = Path.home() / "OneDrive" / "Music"
home_music_dir

WindowsPath('C:/Users/manth/OneDrive/Music')

In [ ]:
import yt_dlp

ydl_opts = {
    "format": "bestaudio",
# 1. Tell yt-dlp to download the thumbnail so we can embed it
    "writethumbnail": True, 
    "playlist_items": "1"
    "postprocessors": [
        # Extract the audio to MP3
        {"key": "FFmpegExtractAudio", "preferredcodec": "mp3"},
        
        # 2. Embed the metadata (Artist, Title, Date, etc.)
        {"key": "FFmpegMetadata"}, 
        
        # 3. Embed the downloaded thumbnail as the album cover art
        {"key": "EmbedThumbnail"},
    ],
    "outtmpl": f"{home_music_dir}/%(title)s.%(ext)s",
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    playlist_url = input("Enter the YouTube playlist URL: ")
    # ydl.download([playlist_url])
    

[youtube:tab] Extracting URL: https://music.youtube.com/playlist?list=PL-TqPRS8uqpRFtvEBqAN-a-E8IwgnD5Xm


[youtube:tab] PL-TqPRS8uqpRFtvEBqAN-a-E8IwgnD5Xm: Downloading webpage
[youtube:tab] PL-TqPRS8uqpRFtvEBqAN-a-E8IwgnD5Xm: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: heartbreak next door
[youtube:tab] PL-TqPRS8uqpRFtvEBqAN-a-E8IwgnD5Xm page 1: Downloading API JSON
[info] Downloading playlist thumbnail 3 ...
[info] Writing playlist thumbnail 3 to: C:\Users\manth\OneDrive\Music\heartbreak next door.jpg
[youtube:tab] Playlist heartbreak next door: Downloading 3 items of 3
[download] Downloading item 1 of 3
[youtube] Extracting URL: https://music.youtube.com/watch?v=1e65rEVbZe4
[youtube] 1e65rEVbZe4: Downloading webpage


[youtube] 1e65rEVbZe4: Downloading android vr player API JSON
[info] 1e65rEVbZe4: Downloading 1 format(s): 251
[info] Downloading video thumbnail 41 ...
[info] Writing video thumbnail 41 to: C:\Users\manth\OneDrive\Music\Shampoo Bottles.webp
[download] Destination: C:\Users\manth\OneDrive\Music\Shampoo Bottles.webm
[download] 100% of    3.66MiB in 00:00:00 at 14.30MiB/s  
[ExtractAudio] Destination: C:\Users\manth\OneDrive\Music\Shampoo Bottles.mp3
Deleting original file C:\Users\manth\OneDrive\Music\Shampoo Bottles.webm (pass -k to keep)
[Metadata] Adding metadata to "C:\Users\manth\OneDrive\Music\Shampoo Bottles.mp3"
[ThumbnailsConvertor] Converting thumbnail "C:\Users\manth\OneDrive\Music\Shampoo Bottles.webp" to png
[EmbedThumbnail] ffmpeg: Adding thumbnail to "C:\Users\manth\OneDrive\Music\Shampoo Bottles.mp3"
[download] Downloading item 2 of 3
[youtube] Extracting URL: https://music.youtube.com/watch?v=jJRKYevf3FE
[youtube] jJRKYevf3FE: Downloading webpage


[youtube] jJRKYevf3FE: Downloading android vr player API JSON
[info] jJRKYevf3FE: Downloading 1 format(s): 251
[info] Downloading video thumbnail 41 ...
[info] Writing video thumbnail 41 to: C:\Users\manth\OneDrive\Music\Good Guy.webp
[download] Destination: C:\Users\manth\OneDrive\Music\Good Guy.webm
[download] 100% of    4.25MiB in 00:00:00 at 6.07MiB/s   
[ExtractAudio] Destination: C:\Users\manth\OneDrive\Music\Good Guy.mp3
Deleting original file C:\Users\manth\OneDrive\Music\Good Guy.webm (pass -k to keep)
[Metadata] Adding metadata to "C:\Users\manth\OneDrive\Music\Good Guy.mp3"
[ThumbnailsConvertor] Converting thumbnail "C:\Users\manth\OneDrive\Music\Good Guy.webp" to png
[EmbedThumbnail] ffmpeg: Adding thumbnail to "C:\Users\manth\OneDrive\Music\Good Guy.mp3"
[download] Downloading item 3 of 3
[youtube] Extracting URL: https://music.youtube.com/watch?v=XzwF8urswkk
[youtube] XzwF8urswkk: Downloading webpage


[youtube] XzwF8urswkk: Downloading android vr player API JSON
[info] XzwF8urswkk: Downloading 1 format(s): 251
[info] Downloading video thumbnail 41 ...
[info] Writing video thumbnail 41 to: C:\Users\manth\OneDrive\Music\Love is a Laserquest.webp
[download] Destination: C:\Users\manth\OneDrive\Music\Love is a Laserquest.webm
[download] 100% of    3.19MiB in 00:00:00 at 13.72MiB/s  
[ExtractAudio] Destination: C:\Users\manth\OneDrive\Music\Love is a Laserquest.mp3
Deleting original file C:\Users\manth\OneDrive\Music\Love is a Laserquest.webm (pass -k to keep)
[Metadata] Adding metadata to "C:\Users\manth\OneDrive\Music\Love is a Laserquest.mp3"
[ThumbnailsConvertor] Converting thumbnail "C:\Users\manth\OneDrive\Music\Love is a Laserquest.webp" to png
[EmbedThumbnail] ffmpeg: Adding thumbnail to "C:\Users\manth\OneDrive\Music\Love is a Laserquest.mp3"
[download] Finished downloading playlist: heartbreak next door


In [20]:
def is_playlist():
    with yt_dlp.YoutubeDL({"quiet": True}) as ydl:
        url = input("Enter YouTube url: ")
        if "playlist" in url:
            return True
        info_dict = ydl.extract_info(url, download=False)
    return (info_dict.get("_type", "video") == "playlist")

is_playlist()

True

### Understanding `mutagen` for writing track

In [24]:
import yt_dlp
from mutagen.id3 import ID3, TRCK, ID3NoHeaderError


class TrackWriterPP(yt_dlp.postprocessor.PostProcessor):
    """
    Custom yt-dlp postprocessor that writes the track number
    to the MP3 file after yt-dlp finishes processing.
    """

    def run(self, info):
        # yt-dlp stores the final file path here
        filepath = info.get("filepath")

        if not filepath:
            print("No filepath found in info dict")
            return [], info

        # track index from playlist
        track = info.get("playlist_index")

        if not track:
            print("No playlist_index found")
            return [], info

        print(f"Writing track number {track} to {filepath}")

        try:
            tags = ID3(filepath)
        except ID3NoHeaderError:
            tags = ID3()

        tags.delall("TRCK")
        tags.add(TRCK(encoding=3, text=str(track)))

        tags.save(filepath, v2_version=3)

        return [], info


ydl_opts = {
    "format": "bestaudio",
    "playlist_items": "1",  # download only first track for testing
    "postprocessors": [
        {"key": "FFmpegExtractAudio", "preferredcodec": "mp3"},
    ],
}


url = "https://music.youtube.com/playlist?list=OLAK5uy_kEN7-JfdfM-5WKhy67kHqJGsjJ29-_HaA"

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.add_post_processor(TrackWriterPP(ydl))
    ydl.download([url])

[youtube:tab] Extracting URL: https://music.youtube.com/playlist?list=OLAK5uy_kEN7-JfdfM-5WKhy67kHqJGsjJ29-_HaA


[youtube:tab] OLAK5uy_kEN7-JfdfM-5WKhy67kHqJGsjJ29-_HaA: Downloading webpage
[youtube:tab] OLAK5uy_kEN7-JfdfM-5WKhy67kHqJGsjJ29-_HaA: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: Album - Tranquility Base Hotel & Casino
[youtube:tab] Playlist Album - Tranquility Base Hotel & Casino: Downloading 1 items of 11
[download] Downloading item 1 of 1
[youtube] Extracting URL: https://music.youtube.com/watch?v=5QqXXjDKNP4
[youtube] 5QqXXjDKNP4: Downloading webpage


[youtube] 5QqXXjDKNP4: Downloading android vr player API JSON
[info] 5QqXXjDKNP4: Downloading 1 format(s): 251
[download] Destination: Star Treatment [5QqXXjDKNP4].webm
[download] 100% of    5.37MiB in 00:00:01 at 4.01MiB/s     
[ExtractAudio] Destination: Star Treatment [5QqXXjDKNP4].mp3
Deleting original file Star Treatment [5QqXXjDKNP4].webm (pass -k to keep)
Writing track number 1 to Star Treatment [5QqXXjDKNP4].mp3
[download] Finished downloading playlist: Album - Tranquility Base Hotel & Casino
